### Import Libraries

In [1]:
import os
import csv
from typing import List, Tuple
import cv2 as cv
import logging
import mediapipe as mp
from tqdm import tqdm
from utils.constant import Constant

2025-05-17 16:36:08.823002: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-17 16:36:08.954676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747474569.004515   39077 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747474569.018026   39077 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747474569.118796   39077 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

### Configure Path

In [2]:
INPUT_DIR = '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/'
NOTHING = 'nothing'
UP = 'up'
LEFT = 'turn_left'
RIGHT = 'turn_right'
DOWN = 'down'
OUT_PUT_DIR = 'model/keypoint_classifier/gesture_3/keypoint.csv'
MIN_DETECTION_CONF = 0.7
STATIC_IMAGE_MODE = True
NORMALIZE = False
MAX_NUM_HAND = 1
CONSTANT = Constant()

print(INPUT_DIR + NOTHING)

/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/nothing


### Configure logging

In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

## Define function

### Initialize MediaPipe Hands model.

In [4]:
def init_hands_model(confidence) -> mp.solutions.hands.Hands:
    return mp.solutions.hands.Hands(
        static_image_mode=STATIC_IMAGE_MODE,
        max_num_hands=MAX_NUM_HAND,
        min_detection_confidence=confidence
    )

In [5]:
def get_image_paths(input_dir: str) -> List[str]:
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    image_paths = []
        
    for root, _, files in os.walk(input_dir):
        for file in files:
            if os.path.splitext(file)[1].lower() in valid_extensions:
                image_paths.append(os.path.join(root, file))
        
    return image_paths

### Process Image Function

In [6]:
def process_image(
    image_path: str,
    hands_model: mp.solutions.hands.Hands,
    class_label: int ,
    normalize: bool
) -> Tuple[List[float], int]:
    
    image = cv.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to read image: {image_path}")

    results = hands_model.process(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    if not results.multi_hand_landmarks:
        raise ValueError("No hands detected")

    hand_landmarks = results.multi_hand_landmarks[0]
    landmarks = []
    for lm in hand_landmarks.landmark:
        if normalize:
            landmarks.extend([lm.x, lm.y])
        else:
            h, w = image.shape[:2]
            landmarks.extend([int(lm.x * w), int(lm.y * h)])

    return landmarks, class_label


## Initialize Hands Model

In [7]:
hands = init_hands_model(MIN_DETECTION_CONF)

I0000 00:00:1747474579.988242   39077 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1747474579.990903   39356 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (TGL GT1)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1747474580.019881   39342 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1747474580.039828   39348 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Load Image Paths

In [8]:
image_paths = get_image_paths(INPUT_DIR+DOWN)

print(image_paths)

['/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-9001dce0-4538-4a9f-9530-b5c101f0e37e.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-836484d6-95dd-4b16-a02d-3070c5f79312.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-85a94338-35f6-466a-acc9-9bc9f92e712b.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-d435e061-d3bc-4d64-b15a-5fd3487c41ce.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-ef1f7125-7d4e-486b-86c2-320d5f1c368f.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-2cfccd24-976c-452d-a035-00112fe88a5e.jpg', '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/down/imgdown-cb1267d9-ae2b-4db5-9a8d-a50

## Process Images and Write to CSV

In [9]:
# ['nothing','down','turn_left','turn_right','up']

file_exists = os.path.isfile(OUT_PUT_DIR) and os.path.getsize(OUT_PUT_DIR) > 0
mode = 'a' if file_exists else 'w'

processed = skipped = 0

with open(OUT_PUT_DIR, mode, newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    CLASS_LABEL = CONSTANT.DOWN
    progress_bar = tqdm(image_paths, desc="Processing images")
     
    for image_path in progress_bar:
        try:
            landmarks, class_id = process_image(
                image_path, hands, CLASS_LABEL , NORMALIZE
            )
            writer.writerow([class_id] + landmarks)
            processed += 1
        except Exception as e:
            logger.warning(f"Skipped {image_path}: {str(e)}")
            skipped += 1

logger.info(f"Processing complete. Success: {processed}, Skipped: {skipped}")
logger.info(f"Dataset saved to {OUT_PUT_DIR}")


Processing images: 100%|██████████| 4000/4000 [01:32<00:00, 43.26it/s]
INFO: Processing complete. Success: 3172, Skipped: 828
INFO: Dataset saved to model/keypoint_classifier/gesture_3/keypoint.csv
